# Phase 7 — Adaptive Behaviour & Feedback Signals
## OpsPilot: Feedback-Driven Prompt Adaptation

**Core idea:** The agent changes its *behaviour* based on how users rate its responses —
without retraining the model. Adaptation lives entirely in the prompt.

| Component | Role |
|-----------|------|
| `FeedbackStore` | Records user ratings (1–5) per query |
| `AdaptiveConfig` | Tracks current style: verbosity, recommendations, uncertainty flags |
| `build_adaptive_agent()` | Rebuilds AgentExecutor with updated style instructions |
| Before/After comparison | Same queries, measurably different response lengths |

**What adapts:**
- **Verbosity**: standard → concise (after low ratings) or → detailed (after high ratings requesting more)
- **Recommendations**: suppress `Recommend:` endings if not wanted
- **Uncertainty flags**: adjust ⚠️ sensitivity based on feedback

In [ ]:
# Cell 1 — Install dependencies
!pip install langchain langchain-openai chromadb openai pandas python-dotenv pysqlite3-binary -q

In [ ]:
# Cell 2 — All imports

# ── SQLite3 patch for ChromaDB on Vocareum ───────────────────────────────────
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

import os
import json
import time
import warnings
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Path setup ───────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR / '../agent').exists() else NOTEBOOK_DIR
for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / 'agent'), str(PROJECT_ROOT / 'data')]:
    if p not in sys.path:
        sys.path.insert(0, p)

from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

print('All imports OK')

In [ ]:
# Cell 3 — API key (Vocareum sets this automatically)
API_KEY = os.environ.get('OPENAI_API_KEY', '')
assert API_KEY, '❌ OPENAI_API_KEY not found in environment.'
print(f'API key ready ✓  (length: {len(API_KEY)} chars)')

In [ ]:
# Cell 4 — Load data and initialise

data_dir    = PROJECT_ROOT / 'data'
incidents   = pd.read_csv(data_dir / 'incidents.csv')
incidents['opened_at'] = pd.to_datetime(incidents['opened_at'])
sla_targets = pd.read_csv(data_dir / 'sla_targets.csv')

collection = None
try:
    import chromadb
    from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
    chroma_client = chromadb.PersistentClient(path=str(data_dir / 'vectorstore'))
    ef = OpenAIEmbeddingFunction(api_key=API_KEY, model_name='text-embedding-3-small')
    collection = chroma_client.get_collection('ops_knowledge', embedding_function=ef)
    print(f'ChromaDB loaded ✓  ({collection.count()} chunks)')
except Exception as e:
    print(f'ChromaDB not available ({e})')

from tool_agent     import init_agent_data
from memory_agent   import SessionMemory
from adaptive_agent import AdaptiveConfig, FeedbackStore, build_adaptive_agent, run_adaptive

init_agent_data(incidents, sla_targets, collection)
print(f'\nData loaded: {len(incidents):,} incidents')
print('Adaptive agent components ready ✓')

In [ ]:
# Cell 5 — Inspect AdaptiveConfig and FeedbackStore defaults

config   = AdaptiveConfig()
feedback = FeedbackStore(log_dir=str(PROJECT_ROOT / 'logs'))

print('AdaptiveConfig (initial state):')
print(f'  verbosity              : {config.verbosity}')
print(f'  include_recommendations: {config.include_recommendations}')
print(f'  uncertainty_sensitivity: {config.uncertainty_sensitivity}')
print(f'  style_instructions()   : "{config.style_instructions() or "(none — standard default)"}"')
print()
print('FeedbackStore:')
print(f'  Summary: {feedback.summary()}')
print(f'  Low threshold  (triggers adapt) : avg < {feedback.LOW_THRESHOLD}')
print(f'  High threshold (restores style) : avg > {feedback.HIGH_THRESHOLD}')
print(f'  Rating window                   : last {feedback.WINDOW} ratings')

In [ ]:
# Cell 6 — BEFORE: Baseline responses in standard (verbose) mode

TEST_QUERIES = [
    'How many P1 incidents has auth-service had in the last 30 days?',
    'What is the SLA breach rate for payments-api?',
    'Give me a health summary of database-cluster.',
]

# Build agent with default (standard) config
executor_before = build_adaptive_agent(API_KEY, config)
memory_before   = SessionMemory(max_turns=10)

before_results = []

print('BEFORE ADAPTATION — Standard Style')
print('='*65)
for q in TEST_QUERIES:
    r = run_adaptive(executor_before, q, memory_before)
    before_results.append(r)
    print(f'\nQ: {q}')
    print(f'Response ({r["word_count"]} words):')
    print(r['response'])
    print(f'Tools: {[tc["tool"] for tc in r["tool_calls"]]}')
    time.sleep(1)

avg_before = sum(r['word_count'] for r in before_results) / len(before_results)
print(f'\nAverage response length (BEFORE): {avg_before:.0f} words')

In [ ]:
# Cell 7 — Simulate user feedback: ratings of 2/5 ("too verbose")

print('USER FEEDBACK — Rating responses 2/5 (too verbose)')
print('='*65)

for i, (q, r) in enumerate(zip(TEST_QUERIES, before_results)):
    feedback.record(
        query     = q,
        response  = r['response'],
        rating    = 2,
        dimension = 'verbosity',
        note      = 'Too long — just need the key number',
    )
    print(f'  Rated {i+1}/3: 2/5 → "{q[:55]}"')

print(f'\nFeedback summary: {feedback.summary()}')

# Apply adaptation
print()
print('Running FeedbackStore.suggest() ...')
change = feedback.suggest(config)
print(f'Adaptation applied: {change}')
print(f'\nAdaptiveConfig after feedback:')
print(f'  verbosity              : {config.verbosity}')
print(f'  style_instructions()   : "{config.style_instructions()}"')

In [ ]:
# Cell 8 — AFTER: Same queries with adapted (concise) config

# Rebuild agent with updated config — this is where the adaptation takes effect
executor_after = build_adaptive_agent(API_KEY, config)   # config.verbosity = 'concise'
memory_after   = SessionMemory(max_turns=10)

after_results = []

print('AFTER ADAPTATION — Concise Style')
print('='*65)
for q in TEST_QUERIES:
    r = run_adaptive(executor_after, q, memory_after)
    after_results.append(r)
    print(f'\nQ: {q}')
    print(f'Response ({r["word_count"]} words):')
    print(r['response'])
    print(f'Tools: {[tc["tool"] for tc in r["tool_calls"]]}')
    time.sleep(1)

avg_after = sum(r['word_count'] for r in after_results) / len(after_results)
print(f'\nAverage response length (AFTER): {avg_after:.0f} words')

In [ ]:
# Cell 9 — Before/After comparison table

print('BEFORE vs AFTER ADAPTATION')
print('='*65)
print(f'{"Query (truncated)":<42} {"Before":>8} {"After":>8} {"Change":>8}')
print('-'*65)

for q, b, a in zip(TEST_QUERIES, before_results, after_results):
    q_short = q[:40] + '..' if len(q) > 42 else q
    diff    = a['word_count'] - b['word_count']
    arrow   = f'{diff:+d} wds'
    print(f'{q_short:<42} {b["word_count"]:>7}w {a["word_count"]:>7}w {arrow:>8}')

print('='*65)
avg_before = sum(r['word_count'] for r in before_results) / len(before_results)
avg_after  = sum(r['word_count'] for r in after_results)  / len(after_results)
reduction  = (1 - avg_after / avg_before) * 100
print(f'{"AVERAGE":<42} {avg_before:>7.0f}w {avg_after:>7.0f}w {-reduction:>+7.0f}%')
print()
print(f'Response length reduced by {reduction:.0f}% after 3 low ratings (2/5).')
print(f'Safety constraints unchanged — tools still called, data still cited.')

In [ ]:
# Cell 10 — Positive feedback loop: high ratings restore standard style

print('POSITIVE FEEDBACK — Rating adapted responses 4/5 ("just right")')
print('='*65)

for q, r in zip(TEST_QUERIES, after_results):
    feedback.record(
        query     = q,
        response  = r['response'],
        rating    = 4,
        dimension = 'verbosity',
        note      = 'Good length',
    )

print(f'Feedback summary: {feedback.summary()}')

# After high ratings, stays concise (avg 4.0 is at the HIGH_THRESHOLD boundary)
# One more 5/5 would push above it
feedback.record(
    query='Is auth-service healthy?',
    response='auth-service: DEGRADED. 2 open incidents.',
    rating=5,
    dimension='verbosity',
    note='Perfect',
)

print(f'After one 5/5 rating: {feedback.summary()}')
print()
change2 = feedback.suggest(config)
print(f'Adaptation applied: {change2}')
print(f'verbosity now: {config.verbosity}')
print()
print('Adaptation log:')
for entry in config.adaptation_log:
    print(f'  [{entry["ts"][:19]}] {entry["msg"]}')

In [ ]:
# Cell 11 — Implicit feedback signals
# Explicit ratings (1-5) aren't always available.
# Show how implicit signals can serve as proxies.

print('IMPLICIT FEEDBACK SIGNALS')
print('='*65)
print()
print('Implicit signal → what it means → adaptive action')
print('-'*65)

IMPLICIT_SIGNALS = [
    ('User immediately asks "can you be shorter?"',
     'Verbosity complaint',
     'decrease verbosity',
     'verbosity'),
    ('User asks same question again after agent answered',
     'Response was unclear or incomplete',
     'switch to detailed mode',
     'verbosity'),
    ('User says "stop with the recommendations"',
     'Recommendation fatigue',
     'suppress Recommend: endings',
     'recommendations'),
    ('User asks "why do you keep saying ⚠️?"',
     'Uncertainty flag overuse',
     'decrease uncertainty sensitivity',
     'uncertainty'),
    ('User asks 3+ follow-ups on same topic',
     'High engagement — wants more detail',
     'increase verbosity',
     'verbosity'),
]

# Simulate processing implicit signals
implicit_config = AdaptiveConfig()   # fresh config

for signal, meaning, action, dimension in IMPLICIT_SIGNALS:
    print(f'Signal  : {signal}')
    print(f'Meaning : {meaning}')
    print(f'Action  : {action}')
    # Map to explicit adaptation
    if 'decrease verbosity' in action:
        change = implicit_config.apply_adaptation('verbosity', 'decrease')
    elif 'detailed' in action:
        change = implicit_config.apply_adaptation('verbosity', 'increase')
    elif 'suppress' in action:
        change = implicit_config.apply_adaptation('recommendations', 'suppress')
    elif 'decrease uncertainty' in action:
        change = implicit_config.apply_adaptation('uncertainty', 'decrease')
    elif 'increase verbosity' in action:
        change = implicit_config.apply_adaptation('verbosity', 'increase')
    else:
        change = 'no change'
    print(f'Config  : {change}')
    print()

print('Implicit config after all signals:')
print(f'  {implicit_config}')
print(f'  Style instructions: "{implicit_config.style_instructions()}"')

In [ ]:
# Cell 12 — Save feedback log
log_path = feedback.save('phase7_feedback.json')
print(f'Feedback log saved → {log_path}')

log_data = json.loads(Path(log_path).read_text())
print(f'Total ratings stored: {log_data["count"]}')
print(f'\nRating history:')
for r in log_data['ratings']:
    print(f'  [{r["ts"][:19]}] {r["rating"]}/5 | {r["dimension"]:12} | {r["query"][:50]}')

In [ ]:
# Cell 13 — Phase 7 Summary

print('PHASE 7 COMPLETE — Adaptive Behaviour & Feedback')
print('='*65)
print()
print('Feedback loop demonstrated:')
print('  1. Baseline responses → user rates 2/5 (too verbose)')
print('  2. FeedbackStore detects avg < 2.5 → triggers adaptation')
print('  3. AdaptiveConfig: verbosity standard → concise')
print('  4. build_adaptive_agent() rebuilds with new style instructions')
print('  5. Same queries → shorter responses → reduction measured')
print('  6. Positive ratings (4/5, 5/5) → style restored')
print()
print(f'Measured response length reduction: {reduction:.0f}%')
print(f'Adaptation log entries: {len(config.adaptation_log)}')
print()
print('What does NOT change with adaptation:')
print('  ✅ Safety rules (read-only, refuse actions) — always enforced')
print('  ✅ Tool routing accuracy — agent still picks correct tools')
print('  ✅ Data accuracy — facts always come from tools, not invented')
print()
print('Known limitations (Phase 7):')
print('  ⚠️  KL13: Config resets between Python sessions (no persistence yet)')
print('  ⚠️  KL14: Adaptation is per-session, not per-analyst')
print('  ⚠️  KL15: Implicit signals require NLP parsing (Phase 9 evaluation)')
print('  ⚠️  KL16: No upper bound on adaptation — could over-correct in loops')
print()
print('Next: Phase 8 — Deployment Readiness (FastAPI, logging, latency)')